# Market Basket Analysis: Hierarchy-Aware Semantic Product Clustering & Association Mining

This notebook implements an NLP & Deep Learning pipeline that groups similar grocery products using Instacart's hierarchical taxonomy (`department` and `aisle`) combined with CUDA GPU-accelerated Sentence Transformers (`all-MiniLM-L6-v2`) and distance-threshold clustering.

### Key Improvements over Global K-Means:
1. **Strict Department Isolation:** Products in different departments (e.g. `produce` vs `personal care`) are never compared or clustered together. "Apple Fruit" and "Apple Shampoo" remain strictly separated.
2. **No Arbitrary K:** Rather than squeezing all products into an arbitrary number of clusters (e.g., $k=120$), clustering uses a semantic **cosine distance threshold** ($\le 0.18$, equivalent to $\ge 82\%$ cosine similarity). Genuine variations cluster naturally while unique products stay as individual entities.
3. **GPU Accelerated:** Embeddings are computed with PyTorch on CUDA in seconds.

In [1]:
# ==========================================
# 0. SETUP & DEPENDENCY IMPORTS
# ==========================================
import os
os.environ["NLTK_DISABLE_IMPORT_SECURITY"] = "1"

import pandas as pd
import numpy as np
import re
import json
import time
from pathlib import Path

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import torch
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch version: {torch.__version__}")
print(f"Compute Device: {device} ({torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'})")


PyTorch version: 2.6.0+cu124
Compute Device: cuda (NVIDIA GeForce RTX 4050 Laptop GPU)


## 1. Data Ingestion & Hierarchical Taxonomy Alignment
We link `products.csv` with `departments.csv` and `aisles.csv`. This ensures every product has explicit category boundaries.

In [2]:
data_dir = Path('../InstacartMBA')
if not data_dir.exists():
    data_dir = Path('InstacartMBA')

products = pd.read_csv(data_dir / 'products.csv')
departments = pd.read_csv(data_dir / 'departments.csv')
aisles = pd.read_csv(data_dir / 'aisles.csv')

product_meta = products.merge(departments, on='department_id', how='left').merge(aisles, on='aisle_id', how='left')

# If merged_df is already defined in the active session, attach department and aisle
if 'merged_df' in globals() or 'merged_df' in locals():
    if 'department' not in merged_df.columns:
        merged_df = merged_df.merge(
            product_meta[['product_id', 'product_name', 'department', 'aisle']],
            on='product_name',
            how='left'
        )
else:
    order_file = data_dir / 'order_products__train.csv'
    if not order_file.exists():
        order_file = data_dir / 'order_products__prior.csv'
    order_products = pd.read_csv(order_file)
    merged_df = order_products.merge(product_meta, on='product_id', how='inner')

# Compute product purchase popularity
item_counts = merged_df['product_name'].value_counts().reset_index()
item_counts.columns = ['product_name', 'purchase_count']

product_catalog = product_meta.merge(item_counts, on='product_name', how='left').fillna({'purchase_count': 0})
product_catalog['purchase_count'] = product_catalog['purchase_count'].astype(int)

print(f"Catalog loaded: {len(product_catalog):,} products across {product_catalog['department'].nunique()} departments and {product_catalog['aisle'].nunique()} aisles.")


Catalog loaded: 49,688 products across 21 departments and 134 aisles.


## 2. Lexical Cleaning & Department-Partitioned Exact Consolidation
We strip packaging sizes, units, retailer branding, and stop words, then lemmatize. Exact match consolidation is partitioned strictly by `(department, cleaned_name)`.

In [3]:
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)
lemmatizer = WordNetLemmatizer()

retail_stop_words = set(stopwords.words('english'))
retail_stop_words.update([
    'organic', 'bag', 'of', 'large', 'small', 'oz', 'pack', 'fresh', 'bunch', 'free', 'gluten',
    'sabra', 'cedars', 'hope', 'stacy', 'kraft', 'chobani', 'trader', 'joes', 'whole', 'foods',
    'value', 'great', 'classic', 'original', 'traditional', 'style', 'size', 'lb', 'ct', 'count'
])

def clean_text(text):
    original = str(text)
    text = original.lower()
    text = re.sub(r'\d+oz|\d+ct|\d+lb|\d+|\b\d+\b', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    words = [lemmatizer.lemmatize(word) for word in text.split() if word not in retail_stop_words]
    cleaned = " ".join(words).strip()
    if not cleaned:
        return re.sub(r'[^a-z0-9\s]', '', original.lower()).strip()
    return cleaned

print("Executing Lexical Deduplication...")
product_catalog['cleaned_name'] = product_catalog['product_name'].apply(clean_text)

print("Consolidating exact matches strictly within each department...")
dept_exact_idx = product_catalog.groupby(['department', 'cleaned_name'])['purchase_count'].idxmax()
most_popular_variants = product_catalog.loc[dept_exact_idx]

exact_match_map = dict(zip(
    zip(most_popular_variants['department'], most_popular_variants['cleaned_name']),
    most_popular_variants['product_name']
))

product_catalog['canonical_group'] = [
    exact_match_map[(dept, cleaned)]
    for dept, cleaned in zip(product_catalog['department'], product_catalog['cleaned_name'])
]
print(f"Canonical groups formed: {product_catalog['canonical_group'].nunique():,} distinct items.")


Executing Lexical Deduplication...


Consolidating exact matches strictly within each department...


Canonical groups formed: 46,369 distinct items.


## 3. GPU Semantic Embeddings & Distance-Threshold Clustering (Per Department)
We encode product names with aisle context using `all-MiniLM-L6-v2` on CUDA. Clustering is executed per department using `AgglomerativeClustering` with `distance_threshold=0.18` (cosine similarity $\ge 0.82$).

In [4]:
print("Initializing SentenceTransformer on GPU...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

canonical_items = product_catalog[['department', 'aisle', 'canonical_group']].drop_duplicates().copy()
canonical_counts = product_catalog.groupby(['department', 'canonical_group'])['purchase_count'].max().reset_index()
canonical_items = canonical_items.merge(canonical_counts, on=['department', 'canonical_group'])

group_to_rep_dict = {}
departments_list = canonical_items['department'].dropna().unique()
print(f"Clustering products across {len(departments_list)} departments using semantic distance threshold...")

COSINE_DISTANCE_THRESHOLD = 0.18
t0 = time.time()

for dept in departments_list:
    dept_subset = canonical_items[canonical_items['department'] == dept].copy().reset_index(drop=True)
    num_items = len(dept_subset)
    
    if num_items <= 1:
        for item in dept_subset['canonical_group']:
            group_to_rep_dict[(dept, item)] = item
        continue

    texts_to_embed = [
        f"{row['aisle']}: {row['canonical_group']}"
        for _, row in dept_subset.iterrows()
    ]
    
    embeddings = embed_model.encode(
        texts_to_embed,
        batch_size=256,
        show_progress_bar=False,
        normalize_embeddings=True
    )
    
    clusterer = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=COSINE_DISTANCE_THRESHOLD,
        metric='cosine',
        linkage='average'
    )
    dept_subset['cluster_id'] = clusterer.fit_predict(embeddings)
    
    rep_idx = dept_subset.groupby('cluster_id')['purchase_count'].idxmax()
    cluster_reps = dict(zip(dept_subset.loc[rep_idx, 'cluster_id'], dept_subset.loc[rep_idx, 'canonical_group']))
    
    for _, row in dept_subset.iterrows():
        rep_name = cluster_reps[row['cluster_id']]
        group_to_rep_dict[(dept, row['canonical_group'])] = rep_name

t1 = time.time()
print(f"Semantic clustering complete in {t1 - t0:.2f}s!")

product_catalog['representative_name'] = [
    group_to_rep_dict.get((dept, can_group), can_group)
    for dept, can_group in zip(product_catalog['department'], product_catalog['canonical_group'])
]

group_to_specific_map = dict(zip(product_catalog['product_name'], product_catalog['representative_name']))

# Export group_mapping.json for Web Backend & Kafka pipeline
out_json_path = Path('../app/group_mapping.json') if Path('../app').exists() else Path('app/group_mapping.json')
with open(out_json_path, 'w', encoding='utf-8') as f:
    json.dump(group_to_specific_map, f, indent=2)

print(f"Group mapping saved to {out_json_path} with {len(group_to_specific_map):,} mappings.")
print(f"Distinct representative entities: {len(set(group_to_specific_map.values())):,}")


Initializing SentenceTransformer on GPU...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Clustering products across 21 departments using semantic distance threshold...


Semantic clustering complete in 47.18s!
Group mapping saved to ..\app\group_mapping.json with 49,688 mappings.
Distinct representative entities: 19,426


## 4. Transaction Mining & FP-Growth Rule Generation
We map customer transactions to the representative entities, run FP-Growth for frequent itemsets, generate directional rules, and filter to clean 1-to-1 association pairs (`antecedent -> consequent`).

In [5]:
print("Preparing transaction arrays for FP-Growth...")
merged_df['product_group'] = merged_df['product_name'].map(group_to_specific_map)

# Focus on active representative groups to manage memory and sparsity
top_groups = merged_df['product_group'].value_counts().head(1500).index
filtered_df = merged_df[merged_df['product_group'].isin(top_groups)]

order_ids = filtered_df['order_id'].drop_duplicates()
if len(order_ids) > 300000:
    sample_order_ids = order_ids.sample(n=300000, random_state=42)
    final_df = filtered_df[filtered_df['order_id'].isin(sample_order_ids)]
else:
    final_df = filtered_df

transactions = final_df.groupby('order_id')['product_group'].apply(list).values

print(f"Running FP-Growth Engine on {len(transactions):,} transactions...")
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
basket_sets = pd.DataFrame(te_ary, columns=te.columns_)

frequent_itemsets = fpgrowth(basket_sets, min_support=0.01, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

# Filter to genuine single-item directional rules (antecedent -> consequent)
rules['antecedents'] = rules['antecedents'].apply(lambda x: list(x)[0] if len(x) > 0 else None)
rules['consequents'] = rules['consequents'].apply(lambda x: list(x)[0] if len(x) > 0 else None)
rules = rules.dropna(subset=['antecedents', 'consequents'])

# Strip self-referencing rules
rules = rules[rules['antecedents'] != rules['consequents']]

final_rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values('lift', ascending=False)

out_rules_path = Path('../app/rules.csv') if Path('../app').exists() else Path('app/rules.csv')
final_rules.to_csv(out_rules_path, index=False)

print(f"Data pipeline complete! Exported {len(final_rules):,} clean, highly distinct rules to {out_rules_path}.")


Preparing transaction arrays for FP-Growth...


Running FP-Growth Engine on 128,882 transactions...


Data pipeline complete! Exported 410 clean, highly distinct rules to ..\app\rules.csv.
